<a href="https://colab.research.google.com/github/snehabenny25/Band-Wise-entropy-Analysis/blob/main/CHBMIT_ThreePhase_Entropy_Trajectory_Postprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CHB-MIT: Seizure-specific three-phase entropy trajectory analysis

This is a **post-processing notebook** for the existing all-band entropy extraction notebook. It does not filter EEG again or recalculate PE/WPE/MPE.

## Purpose
For each annotated seizure:
1. Identify the nearest preceding **seizure-free EDF from the same patient's ordered CHB-MIT summary**.
2. Select the final 30 minutes of that EDF as the seizure-associated interictal reference.
3. Link that baseline to the seizure's existing 45-minute preictal and ictal window-level entropy observations.
4. Save a seizure-specific three-phase dataset and plot raw window-level trajectories.

**Important:** The interictal and seizure recordings are separate EDFs and may have a recording gap. The plotting function therefore uses three aligned panels rather than drawing a misleading continuous line across that gap. Every valid window remains an individual observation; no trajectory averaging is used.

## Existing files expected
The default input is:

`/content/drive/MyDrive/chbmit/entropy_frequency_band_results_overlapping_allbands_v3/window_level_entropy_trajectories.csv`

The notebook can alternatively rebuild the long-format input from the `per_edf/*_entropy.csv` files.

## Before running
- Let the original extraction notebook finish writing its outputs.
- Confirm the `OUTPUT_DIR` below matches the folder used by that notebook.
- Do not delete or overwrite the original extraction outputs.


## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_ROOT = Path('/content/drive/MyDrive/chbmit/1.0.0')
OUTPUT_DIR = Path('/content/drive/MyDrive/chbmit/entropy_frequency_band_results_overlapping_allbands_v3')
PER_FILE_DIR = OUTPUT_DIR / 'per_edf'

SUMMARY_DIR = OUTPUT_DIR / 'three_phase_entropy_analysis'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

INTERICTAL_MINUTES = 30
PREICTAL_MINUTES = 45
METRICS = ['PE', 'WPE', 'MPE_s1', 'MPE_s2', 'MPE_s3', 'MPE_s4', 'MPE_s5']

LONG_FILE = OUTPUT_DIR / 'window_level_entropy_trajectories.csv'

print('Dataset root exists:', DATA_ROOT.exists())
print('Output folder exists:', OUTPUT_DIR.exists())
print('Long-format input exists:', LONG_FILE.exists())
print('Per-EDF folder exists:', PER_FILE_DIR.exists())


## 2. Load existing entropy observations (no recalculation)

In [ ]:
required_cols = {
    'patient_id','file_name','channel','window_start_sec','window_end_sec',
    'window_mid_sec','band','condition','metric','entropy'
}

if LONG_FILE.exists():
    long = pd.read_csv(LONG_FILE)
    missing = required_cols - set(long.columns)
    if missing:
        raise ValueError(f'Long-format file is missing required columns: {sorted(missing)}')
    print('Loaded existing long-format trajectory file:', LONG_FILE)
else:
    # Fallback: combine per-EDF feature CSVs and reshape existing entropy columns.
    if not PER_FILE_DIR.exists():
        raise FileNotFoundError('Neither the long-format file nor per_edf folder is available.')
    files = sorted(PER_FILE_DIR.glob('*_entropy.csv'))
    if not files:
        raise FileNotFoundError('No per-EDF entropy CSVs found. Wait for the extraction notebook to produce them.')
    feature_parts = []
    for f in files:
        try:
            d = pd.read_csv(f)
            if not d.empty:
                feature_parts.append(d)
        except Exception as exc:
            print('Could not read', f.name, ':', exc)
    if not feature_parts:
        raise RuntimeError('No readable per-EDF entropy rows were found.')
    features = pd.concat(feature_parts, ignore_index=True)
    value_metrics = [m for m in METRICS if m in features.columns]
    id_cols = [c for c in [
        'patient_id','edf_file','file_name','channel','window_idx',
        'window_start_sec','window_end_sec','window_mid_sec','band',
        'condition','seizure_start_sec','seizure_end_sec',
        'relative_to_seizure_onset_sec','time_to_onset_min'
    ] if c in features.columns]
    long = features.melt(id_vars=id_cols, value_vars=value_metrics,
                         var_name='metric', value_name='entropy').dropna(subset=['entropy'])
    long['seizure_id'] = np.where(
        long['condition'].isin(['preictal','ictal']),
        long['file_name'].astype(str) + '__onset_' +
        long['seizure_start_sec'].round(3).astype(str),
        ''
    )
    long.to_csv(SUMMARY_DIR / 'rebuilt_long_entropy_input.csv', index=False)
    print('Rebuilt long-format data from existing per-EDF feature CSVs.')

# Normalize common data types and check required information.
long['patient_id'] = long['patient_id'].astype(str).str.lower()
long['file_name'] = long['file_name'].astype(str)
long['condition'] = long['condition'].astype(str).str.lower()
long['metric'] = long['metric'].astype(str)
long['band'] = long['band'].astype(str)
long['entropy'] = pd.to_numeric(long['entropy'], errors='coerce')
for c in ['window_start_sec','window_end_sec','window_mid_sec']:
    long[c] = pd.to_numeric(long[c], errors='coerce')
long = long.dropna(subset=['entropy','window_start_sec','window_end_sec','window_mid_sec'])
print('Rows:', len(long))
print('Patients:', long.patient_id.nunique())
print('Conditions:', sorted(long.condition.unique()))
print('Bands:', sorted(long.band.unique()))
print('Metrics:', sorted(long.metric.unique()))
display(long.head())


## 3. Read EDF chronology and seizure annotations from CHB-MIT summary files

The summary file's `File Name:` order is used as the within-patient recording order. For each seizure-containing EDF, the code searches backwards through that patient's ordered EDF list and selects the closest preceding EDF with **no annotated seizure events**.

If a summary file is unavailable, a file-order fallback based on EDF filenames is used only to create an audit warning; inspect those cases before relying on them.


In [ ]:
def patient_id_from_name(name):
    m = re.search(r'(chb\\d+)', str(name), re.I)
    return m.group(1).lower() if m else 'unknown'

def parse_summary_in_order(summary_path):
    text = summary_path.read_text(errors='ignore')
    records = []
    current = None
    starts, ends = [], []

    def flush():
        nonlocal current, starts, ends
        if current:
            intervals = []
            for i, start in enumerate(starts):
                if i < len(ends) and ends[i] >= start:
                    intervals.append((float(start), float(ends[i])))
            records.append({'file_name': Path(current).name,
                            'events': intervals})
        current, starts, ends = None, [], []

    for line in text.splitlines():
        fm = re.search(r'File Name:\\s*(\\S+\\.edf)', line, re.I)
        if fm:
            flush()
            current = fm.group(1).strip()
            continue
        sm = re.search(r'Seizure\\s*\\d*\\s*Start Time:\\s*(\\d+(?:\\.\\d+)?)\\s*seconds?', line, re.I)
        em = re.search(r'Seizure\\s*\\d*\\s*End Time:\\s*(\\d+(?:\\.\\d+)?)\\s*seconds?', line, re.I)
        if sm: starts.append(float(sm.group(1)))
        if em: ends.append(float(em.group(1)))
    flush()
    return records

summary_files = sorted(DATA_ROOT.rglob('*summary*.txt')) if DATA_ROOT.exists() else []
ordered_by_patient = {}
summary_source = {}
for sf in summary_files:
    pid = patient_id_from_name(sf.name)
    if pid == 'unknown':
        continue
    recs = parse_summary_in_order(sf)
    if recs:
        ordered_by_patient[pid] = recs
        summary_source[pid] = str(sf)

# Build a file-to-events lookup from the ordered summary records.
events_by_patient_file = {}
for pid, recs in ordered_by_patient.items():
    for rec in recs:
        events_by_patient_file[(pid, rec['file_name'].lower())] = rec['events']

print('Patients with ordered summary chronology:', len(ordered_by_patient))
print('Summary files found:', len(summary_files))
for pid, recs in list(ordered_by_patient.items())[:5]:
    print(pid, '| EDF entries:', len(recs), '| first:', recs[0]['file_name'], '| last:', recs[-1]['file_name'])


## 4. Build seizure-specific three-phase datasets

In [ ]:
# Identify seizure IDs present in the already-computed preictal/ictal data.
if 'seizure_id' not in long.columns:
    if 'seizure_start_sec' not in long.columns:
        raise ValueError('No seizure_id or seizure_start_sec column is available.')
    long['seizure_id'] = np.where(
        long.condition.isin(['preictal','ictal']),
        long.file_name.astype(str) + '__onset_' +
        pd.to_numeric(long.seizure_start_sec, errors='coerce').round(3).astype(str),
        ''
    )

seizure_rows = long[long.condition.isin(['preictal','ictal'])].copy()
seizure_rows = seizure_rows[seizure_rows.seizure_id.notna() & seizure_rows.seizure_id.ne('')]

# One event record per patient, seizure EDF and seizure ID.
event_table = (seizure_rows[['patient_id','file_name','seizure_id']]
               .drop_duplicates()
               .sort_values(['patient_id','file_name','seizure_id'])
               .reset_index(drop=True))

def get_previous_seizure_free_edf(pid, seizure_file):
    recs = ordered_by_patient.get(pid, [])
    names = [r['file_name'].lower() for r in recs]
    target = str(seizure_file).lower()
    if target not in names:
        return None, 'seizure EDF absent from summary chronology'
    idx = names.index(target)
    for j in range(idx - 1, -1, -1):
        rec = recs[j]
        if len(rec['events']) == 0:
            return rec['file_name'], 'summary chronology'
    return None, 'no preceding seizure-free EDF found'

# If there is no summary chronology for a patient, do not silently guess.
# Such events are recorded in the audit and excluded from matched three-phase data.
audit_rows = []
matched_parts = []

for ev in event_table.itertuples(index=False):
    pid, seizure_file, sid = ev.patient_id, ev.file_name, ev.seizure_id
    baseline_file, method = get_previous_seizure_free_edf(pid, seizure_file)

    if baseline_file is None:
        audit_rows.append({'patient_id':pid,'seizure_file':seizure_file,'seizure_id':sid,
                           'baseline_file':'','status':'NO_BASELINE','selection_method':method,
                           'baseline_duration_sec':np.nan,'baseline_windows':0})
        continue

    baseline = long[(long.patient_id == pid) &
                    (long.file_name.str.lower() == baseline_file.lower()) &
                    (long.condition == 'interictal')].copy()
    if baseline.empty:
        audit_rows.append({'patient_id':pid,'seizure_file':seizure_file,'seizure_id':sid,
                           'baseline_file':baseline_file,'status':'BASELINE_FEATURES_MISSING',
                           'selection_method':method,'baseline_duration_sec':np.nan,'baseline_windows':0})
        continue

    # Estimate EDF duration from the largest recorded window endpoint in its extracted feature rows.
    duration_sec = float(baseline.window_end_sec.max())
    required_sec = INTERICTAL_MINUTES * 60
    if duration_sec < required_sec:
        audit_rows.append({'patient_id':pid,'seizure_file':seizure_file,'seizure_id':sid,
                           'baseline_file':baseline_file,'status':'BASELINE_SHORTER_THAN_30_MIN',
                           'selection_method':method,'baseline_duration_sec':duration_sec,'baseline_windows':0})
        continue

    cutoff = duration_sec - required_sec
    baseline = baseline[(baseline.window_start_sec >= cutoff) &
                        (baseline.window_end_sec <= duration_sec)].copy()
    if baseline.empty:
        audit_rows.append({'patient_id':pid,'seizure_file':seizure_file,'seizure_id':sid,
                           'baseline_file':baseline_file,'status':'NO_WINDOWS_IN_FINAL_30_MIN',
                           'selection_method':method,'baseline_duration_sec':duration_sec,'baseline_windows':0})
        continue

    baseline['phase'] = 'interictal'
    baseline['associated_seizure_id'] = sid
    baseline['associated_seizure_file'] = seizure_file
    baseline['baseline_file'] = baseline_file
    baseline['phase_time_min'] = (baseline.window_mid_sec - cutoff) / 60.0

    seizure_part = seizure_rows[(seizure_rows.patient_id == pid) &
                                (seizure_rows.seizure_id == sid)].copy()
    seizure_part['phase'] = seizure_part['condition']
    seizure_part['associated_seizure_id'] = sid
    seizure_part['associated_seizure_file'] = seizure_file
    seizure_part['baseline_file'] = baseline_file
    # Preictal x-axis: negative minutes relative to seizure onset; ictal x-axis: minutes after onset.
    seizure_part['phase_time_min'] = np.where(
        seizure_part.condition == 'preictal',
        pd.to_numeric(seizure_part.get('relative_to_seizure_onset_sec'), errors='coerce') / 60.0,
        pd.to_numeric(seizure_part.get('relative_to_seizure_onset_sec'), errors='coerce') / 60.0
    )

    matched_parts.extend([baseline, seizure_part])
    audit_rows.append({'patient_id':pid,'seizure_file':seizure_file,'seizure_id':sid,
                       'baseline_file':baseline_file,'status':'MATCHED',
                       'selection_method':method,'baseline_duration_sec':duration_sec,
                       'baseline_windows':baseline[['window_idx']].drop_duplicates().shape[0] if 'window_idx' in baseline else np.nan})

audit = pd.DataFrame(audit_rows)
audit.to_csv(SUMMARY_DIR / 'seizure_interictal_baseline_audit.csv', index=False)
display(audit['status'].value_counts(dropna=False).to_frame('seizure_count'))
display(audit.head(20))

if not matched_parts:
    raise RuntimeError('No matched three-phase seizure datasets were created. Inspect the baseline audit and input files.')

three_phase = pd.concat(matched_parts, ignore_index=True, sort=False)
three_phase.to_csv(SUMMARY_DIR / 'seizure_specific_three_phase_entropy_trajectories.csv', index=False)
print('Saved matched three-phase observations:', len(three_phase))
print('Output:', SUMMARY_DIR / 'seizure_specific_three_phase_entropy_trajectories.csv')


## 5. Validate phase coverage and feature availability

The audit is important: only events marked `MATCHED` have a valid preceding seizure-free EDF with at least 30 minutes of extracted interictal windows. Events with missing or short baselines are not silently substituted with unrelated recordings.


In [ ]:
phase_coverage = (three_phase.groupby(['patient_id','associated_seizure_id','phase','band','metric'])
                  .agg(n_window_channel_observations=('entropy','size'),
                       n_channels=('channel','nunique'),
                       first_phase_time_min=('phase_time_min','min'),
                       last_phase_time_min=('phase_time_min','max'))
                  .reset_index())
phase_coverage.to_csv(SUMMARY_DIR / 'three_phase_coverage_by_seizure_band_metric.csv', index=False)

phase_summary = (three_phase.groupby(['phase','band','metric'])
                 .agg(n_observations=('entropy','size'),
                      n_patients=('patient_id','nunique'),
                      n_seizures=('associated_seizure_id','nunique'),
                      mean_entropy=('entropy','mean'),
                      median_entropy=('entropy','median'),
                      sd_entropy=('entropy','std'))
                 .reset_index())
phase_summary.to_csv(SUMMARY_DIR / 'descriptive_three_phase_entropy_summary.csv', index=False)

display(phase_summary.head(30))
display(phase_coverage.head(20))


## 6. Plot one seizure: three aligned phase panels, every window retained

The three panels share the same y-axis scale. Interictal time is measured within the selected final-30-minute baseline EDF; preictal time is relative to seizure onset; ictal time is elapsed time after onset. The panels are deliberately separated because EDF recordings may have a gap.


In [ ]:
def plot_three_phase_trajectory(patient_id, seizure_id, channel, metric='PE', band='delta',
                                save=True, show=True):
    d = three_phase[
        (three_phase.patient_id.astype(str).str.lower() == str(patient_id).lower()) &
        (three_phase.associated_seizure_id == seizure_id) &
        (three_phase.channel == channel) &
        (three_phase.metric == metric) &
        (three_phase.band == band)
    ].copy()

    if d.empty:
        print('No observations for this patient/seizure/channel/metric/band combination.')
        return None

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
    configs = [
        ('interictal', 'Interictal baseline', 'Minutes within final 30-min EDF segment'),
        ('preictal', 'Preictal', 'Minutes relative to seizure onset'),
        ('ictal', 'Ictal', 'Minutes after seizure onset')
    ]

    for ax, (phase, title, xlabel) in zip(axes, configs):
        q = d[d.phase == phase].copy()
        if q.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        else:
            if phase == 'interictal':
                q = q.sort_values('phase_time_min')
                x = q.phase_time_min
            elif phase == 'preictal':
                q = q.sort_values('phase_time_min')
                x = q.phase_time_min
            else:
                q = q.sort_values('phase_time_min')
                x = q.phase_time_min
            # Each marker is an individual window observation; no averaging or smoothing.
            ax.plot(x, q.entropy, marker='.', linestyle='-', linewidth=0.7, markersize=3)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.grid(True, alpha=0.25)

    axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[2].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_ylabel(f'{metric} entropy')
    fig.suptitle(f'{patient_id} | {seizure_id} | {channel} | {band} | {metric}\n'
                 'Three-phase window-level trajectory (panels separated; not a continuous recording)')
    fig.tight_layout()

    if save:
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', f'{patient_id}_{seizure_id}_{channel}_{band}_{metric}')
        fig.savefig(SUMMARY_DIR / f'{safe}_three_phase.png', dpi=220, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig

# Show available matched seizure IDs and example channels.
available = (three_phase[['patient_id','associated_seizure_id','channel','band','metric']]
             .drop_duplicates().sort_values(['patient_id','associated_seizure_id']))
display(available.head(30))

print("Example call:")
print("plot_three_phase_trajectory('chb01', '<associated_seizure_id>', 'FP1-F7', metric='PE', band='theta')")


## 7. Generate plots for all metrics and bands for one selected seizure/channel

In [ ]:
def plot_all_entropy_features_for_seizure(patient_id, seizure_id, channel, save=True):
    subset = three_phase[
        (three_phase.patient_id.astype(str).str.lower() == str(patient_id).lower()) &
        (three_phase.associated_seizure_id == seizure_id) &
        (three_phase.channel == channel)
    ]
    combos = subset[['band','metric']].drop_duplicates().sort_values(['band','metric'])
    print('Figures to generate:', len(combos))
    for row in combos.itertuples(index=False):
        plot_three_phase_trajectory(patient_id, seizure_id, channel,
                                    metric=row.metric, band=row.band,
                                    save=save, show=True)

# Example: after inspecting `available`, call:
# plot_all_entropy_features_for_seizure('chb01', '<associated_seizure_id>', 'FP1-F7')


## 8. Outputs and interpretation safeguards

Files saved in `three_phase_entropy_analysis/`:

- `seizure_interictal_baseline_audit.csv` — which baseline EDF was selected for every seizure and why any event was excluded.
- `seizure_specific_three_phase_entropy_trajectories.csv` — all matched interictal, preictal and ictal window-level observations.
- `three_phase_coverage_by_seizure_band_metric.csv` — coverage by seizure, band and entropy metric.
- `descriptive_three_phase_entropy_summary.csv` — descriptive statistics, not a replacement for raw trajectories.
- PNG figures — individual seizure/channel/band/metric three-phase plots.

### Interpretation cautions
1. This post-processing does not recalculate PE, WPE, MPE, KL, JSD, Wasserstein distance or CUSUM.
2. A seizure is included in the matched three-phase dataset only if its preceding seizure-free EDF is found in the summary chronology and has at least 30 minutes of extracted windows.
3. The selected interictal EDF is a temporally preceding reference, not proof of continuous physiological recording up to the preictal EDF.
4. The 10-second windows with a 5-second step overlap. They are correlated observations and must not be treated as independent samples in inferential statistics.
5. Do not conclude that an entropy measure is a validated seizure biomarker from trajectory plots alone. Assess consistency across seizures/patients and predictive performance/false alarms in a separate validation design.
6. The notebook relies on the original extraction notebook's labels and entropy values. It does not repair missing, incomplete or incorrectly labelled source data.
